# Reward-cell classification

Classifies hidden units as reward cells using the shuffle-control method from
Yaghoubi et al. (2026): for each cell, mean activity while the agent is near
a reward location is compared against a null distribution built from 1000
circular shifts of its activity timeseries. A cell is a reward cell if its
true near-reward mean exceeds the 99th percentile of that null distribution.

This also runs the cell-type classification pipeline (`calculate_metrics` +
`groupCells`) so reward-cell rates can be broken down by cell type
(single-field / complex / complex multi-peak).

Two demo networks (see notebook 1 for details): a no-reward baseline and a
reward condition (magnitude x5) with the same seed.

In [1]:
%matplotlib inline
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from prnn.utils.predictiveNet import PredictiveNet
import spatial_analysis as sa

DATA_DIR = REPO_ROOT / "data" / "trajectory"

OBS = [torch.tensor(np.load(DATA_DIR / f"obs{i}.npy")) for i in range(4)]
ACT = torch.tensor(np.load(DATA_DIR / "act.npy"))
STATE = {
    "agent_pos": torch.tensor(np.load(DATA_DIR / "pos.npy")),
    "agent_dir": torch.tensor(np.load(DATA_DIR / "dir.npy")),
    "mean_vel": 0.2,
}

wandb not installed, will not log to wandb


## 1. Compute place fields, classify cell types, then classify reward cells

In [2]:
def analyze(netname, savename, rng_seed=42):
    net = PredictiveNet.loadNet(savename, savefolder=str(REPO_ROOT) + "/", dil=False)
    env = net.EnvLibrary[0]

    place_fields, SI, WAKE = sa.calculateSpatialRepresentation(net, env, OBS, ACT, STATE, saveTrainingData=False)
    metrics = sa.calculate_metrics(env, place_fields, SI, WAKE)
    groups, groupID = sa.groupCells(metrics)

    h_mean = WAKE["h"]
    positions = STATE["agent_pos"].numpy().astype(float)
    positions_norm = positions / positions.max(axis=0)
    reward_pos = env.reward_positions

    is_rew, true_mean, thresh, near_mask = sa.classify_reward_cells(
        h_mean, positions_norm, reward_pos,
        radius=0.05, n_shuffles=1000, percentile=99,
        rng=np.random.default_rng(rng_seed),
    )
    return dict(net=net, env=env, groups=groups, is_rew=is_rew, near_mask=near_mask)


demo = {
    "base_no_reward": analyze("base_no_reward", "base_no_reward_s1001_ep5"),
    "random_rew_mult5": analyze("random_rew_mult5", "random_rew_mult5_s1001_ep5"),
}

for name, res in demo.items():
    is_rew = res["is_rew"]
    print(f"{name}: {is_rew.sum()} / {len(is_rew)} reward cells "
          f"({100*is_rew.mean():.1f}%), agent near reward for {res['near_mask'].sum()} timesteps")

Net Loaded from pathname


/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-packages/pynapple/core/utils.py:114: UserWarning: Converting 'd' to numpy.array. The provided array was of type 'Tensor'.
  warnings.warn(


/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-packages/pynapple/core/utils.py:114: UserWarning: Converting 'd' to numpy.array. The provided array was of type 'Tensor'.
  warnings.warn(


Net Loaded from pathname


/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-packages/pynapple/core/utils.py:114: UserWarning: Converting 'd' to numpy.array. The provided array was of type 'Tensor'.
  warnings.warn(
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-packages/pynapple/process/tuning_curves.py:378: RuntimeWarning: invalid value encountered in true_divide
  fxfr = fx / fr
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-packages/pynapple/process/tuning_curves.py:389: RuntimeWarning: invalid value encountered in true_divide
  SI = SI / fr[:, 0, 0]
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-packages/pynapple/core/utils.py:114: UserWarning: Converting 'd' to numpy.array. The provided array was of type 'Tensor'.
  warnings.warn(
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-packages/pynapple/process/tuning_curves.py:290: Run

/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/prnn/analysis/TuningCurveAnalysis.py:448: RuntimeWarning: invalid value encountered in double_scalars
  border_score = (mean_border_rate - mean_nonborder_rate) / (mean_border_rate + mean_nonborder_rate)


/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/prnn/analysis/TuningCurveAnalysis.py:501: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  centerlabeled = labeled == whicharea
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/prnn/analysis/TuningCurveAnalysis.py:501: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  centerlabeled = labeled == whicharea
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/prnn/analysis/TuningCurveAnalysis.py:510: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  centerlabeled = labeled == whicharea
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/prnn/analysis/TuningCurveAnalysis.py:510: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  centerlabeled = labeled == whicharea
/Users/hadrienpadilla/Documents/McGill/Peyrache Lab/pRNN/.newenv/lib/python3.9/site-

base_no_reward: 193 / 700 reward cells (27.6%), agent near reward for 610 timesteps
random_rew_mult5: 200 / 700 reward cells (28.6%), agent near reward for 610 timesteps


## 2. Breakdown by cell type

In [3]:
for name, res in demo.items():
    print(f"\n=== {name} ===")
    header = f"{'Cell type':<16} {'N cells':>8} {'N reward':>10} {'% reward':>10}"
    print(header)
    print("-" * len(header))
    for gname, mask in res["groups"].items():
        n_type = int(mask.sum())
        n_rew = int((mask & res["is_rew"]).sum())
        pct = 100 * n_rew / n_type if n_type > 0 else float("nan")
        print(f"{gname:<16} {n_type:>8} {n_rew:>10} {pct:>9.1f}%")


=== base_no_reward ===
Cell type         N cells   N reward   % reward
-----------------------------------------------
untuned                 2          0       0.0%
HD_cells                3          0       0.0%
single_field          120         37      30.8%
border_cells            1          0       0.0%
spatial_HD             25          5      20.0%
complex_cells         549        151      27.5%

=== random_rew_mult5 ===
Cell type         N cells   N reward   % reward
-----------------------------------------------
untuned                 3          0       0.0%
HD_cells                0          0       nan%
single_field          101         35      34.7%
border_cells            0          0       nan%
spatial_HD             65         14      21.5%
complex_cells         531        151      28.4%
